#1: scraping one race
Goal: download the results for one GS women's race and turn them into a table
Race: codex 6284, 10 January 2026 (FIS raceid 131070)

In [ ]:
#download race
import requests
from bs4 import BeautifulSoup

url = "https://www.fis-ski.com/DB/general/results.html?sectorcode=AL&raceid=131070"
headers = {"User-Agent": "Mozilla/5.0 (student project: ski-race-analysis)"}

response = requests.get(url, headers=headers, timeout=30)
print(response.status_code)

200


In [ ]:
#save copy
with open("../data/raw/race_131070.html", "w", encoding="utf-8") as f:
    f.write(response.text)

In [ ]:
#find rows, 44 rows
soup = BeautifulSoup(response.text, "html.parser")
rows = soup.select("a.table-row")
print(len(rows))
print(rows[0].get_text(" | ", strip=True))

44
Homologation Number | 12346/02/17


In [ ]:
#look at rows
for i, row in enumerate(rows):
    print(i, row.get("href"), "→", row.get_text(" | ", strip=True)[:80])

0 https://www.fis-ski.com/DB/general/homologations.html?sectorcode=AL&homologationcode=12346%2F02%2F17&homologationtype=DET → Homologation Number | 12346/02/17
1 https://www.fis-ski.com/DB/general/athlete-biography.html?sectorcode=al&competitorid=264785 → 1 | 6 | 426507 | GREIBROKK Mathilde | 2005 | NOR | 57.05 | 59.03 | 1:56.08 | 1:5
2 https://www.fis-ski.com/DB/general/athlete-biography.html?sectorcode=al&competitorid=280252 → 2 | 9 | 426564 | JOHANSEN Eva Unhjem | 2007 | NOR | 58.24 | 57.98 | 1:56.22 | +0
3 https://www.fis-ski.com/DB/general/athlete-biography.html?sectorcode=al&competitorid=264762 → 3 | 13 | 426492 | STEINSVIK Adriane | 2005 | NOR | 58.08 | 58.38 | 1:56.46 | +0.
4 https://www.fis-ski.com/DB/general/athlete-biography.html?sectorcode=al&competitorid=287750 → 4 | 2 | 426615 | BARMEN Emely Karin | 2008 | NOR | 58.08 | 58.66 | 1:56.74 | +0.
5 https://www.fis-ski.com/DB/general/athlete-biography.html?sectorcode=al&competitorid=273073 → 5 | 7 | 426538 | KALDHUSSAETER Frida

In [7]:
#keep only skier rows (trying athlete rows)
athlete_rows = [row for row in rows if row.get("href") and "athlete" in row.get("href")]
print(len(athlete_rows))
print(athlete_rows[0].get_text(" | ", strip=True))

43
1 | 6 | 426507 | GREIBROKK Mathilde | 2005 | NOR | 57.05 | 59.03 | 1:56.08 | 1:56.08 | 29.63 | 37.63


read of results: rank 1, bib 6, FIS code 426507, winner Mathilde Greibrokk, birth year 2005, nationality, run times and points. 

In [8]:
from collections import Counter

parsed = [list(row.stripped_strings) for row in athlete_rows]
print(Counter(len(p) for p in parsed))

Counter({12: 27, 5: 16})


27 finishers, 16 dnf

In [10]:
# a non-finisher
short_rows = [p for p in parsed if len(p) == 5]
print(short_rows[0])

['41', '255508', 'HARDARDOTTIR Viktoria Bjork', '2009', 'ISL']


In [11]:
#finishers
import pandas as pd

columns = ["rank", "bib", "fis_code", "name", "birth_year", "nation",
           "run1", "run2", "total", "diff", "race_points", "points_2"]
finishers = pd.DataFrame([p for p in parsed if len(p) == 12], columns=columns)
finishers.head()


,rank,bib,fis_code,name,birth_year,nation,run1,run2,total,diff,race_points,points_2
0,1,6,426507,GREIBROKK Mathilde,2005,NOR,57.05,59.03,1:56.08,1:56.08,29.63,37.63
1,2,9,426564,JOHANSEN Eva Unhjem,2007,NOR,58.24,57.98,1:56.22,+0.14,30.85,38.85
2,3,13,426492,STEINSVIK Adriane,2005,NOR,58.08,58.38,1:56.46,+0.38,32.94,40.94
3,4,2,426615,BARMEN Emely Karin,2008,NOR,58.08,58.66,1:56.74,+0.66,35.37,43.37
4,5,7,426538,KALDHUSSAETER Frida,2006,NOR,58.16,58.75,1:56.91,+0.83,36.85,44.85


Yay first table

In [ ]:
# finishers status
import re

status_pattern = re.compile(r"did not|disqualified|not permitted", re.IGNORECASE)

def get_status(row):
    heading = row.find_previous(string=status_pattern)
    return heading.strip() if heading else "Finished"

statuses = [get_status(row) for row in athlete_rows]
print(Counter(statuses))

Counter({'Finished': 27, 'Did not finish 2nd run': 11, 'Did not start 1st run': 3, 'Did not finish 1st run': 2})


In [16]:
# table with all racers
records = []
for p, status in zip(parsed, statuses):
    if len(p) == 12:
        record = dict(zip(columns, p))
    else:
        record = dict(zip(["bib", "fis_code", "name", "birth_year", "nation"], p))
    record["status"] = status
    records.append(record)

results = pd.DataFrame(records)
results.tail()

,rank,bib,fis_code,name,birth_year,nation,run1,run2,total,diff,race_points,points_2,status
38,NaN,36,255485,GAUTADOTTIR Snaedis Gudrun,2007,ISL,NaN,NaN,NaN,NaN,NaN,NaN,Did not finish 1st run
39,NaN,23,426630,ENG Thea Lindstroem,2009,NOR,NaN,NaN,NaN,NaN,NaN,NaN,Did not finish 1st run
40,NaN,42,426636,TORKILDSEN Idun Lambert,2009,NOR,NaN,NaN,NaN,NaN,NaN,NaN,Did not start 1st run
41,NaN,26,426633,DYBWAD Sarah,2009,NOR,NaN,NaN,NaN,NaN,NaN,NaN,Did not start 1st run
42,NaN,21,426626,BRUNBORG Nora,2009,NOR,NaN,NaN,NaN,NaN,NaN,NaN,Did not start 1st run


In [17]:
#convert times to seconds
def to_seconds(t):
    if pd.isna(t):
        return None
    if ":" in t:
        minutes, seconds = t.split(":")
        return int(minutes) * 60 + float(seconds)
    return float(t)

for col in ["run1", "run2", "total"]:
    results[col] = results[col].apply(to_seconds)

results["bib"] = results["bib"].astype(int)
results.head()

,rank,bib,fis_code,name,birth_year,nation,run1,run2,total,diff,race_points,points_2,status
0,1,6,426507,GREIBROKK Mathilde,2005,NOR,57.05,59.03,116.08,1:56.08,29.63,37.63,Finished
1,2,9,426564,JOHANSEN Eva Unhjem,2007,NOR,58.24,57.98,116.22,+0.14,30.85,38.85,Finished
2,3,13,426492,STEINSVIK Adriane,2005,NOR,58.08,58.38,116.46,+0.38,32.94,40.94,Finished
3,4,2,426615,BARMEN Emely Karin,2008,NOR,58.08,58.66,116.74,+0.66,35.37,43.37,Finished
4,5,7,426538,KALDHUSSAETER Frida,2006,NOR,58.16,58.75,116.91,+0.83,36.85,44.85,Finished


In [18]:
#find how far behind each racer is from the winner in %
winner_time = results["total"].min()
results["pct_behind"] = (results["total"] / winner_time - 1) * 100
results[["rank", "bib", "name", "total", "pct_behind"]].head()

,rank,bib,name,total,pct_behind
0,1,6,GREIBROKK Mathilde,116.08,0.000000
1,2,9,JOHANSEN Eva Unhjem,116.22,0.120606
2,3,13,STEINSVIK Adriane,116.46,0.327360
3,4,2,BARMEN Emely Karin,116.74,0.568573
4,5,7,KALDHUSSAETER Frida,116.91,0.715024
